# 第三周教师参考答案
依赖Python 3 + NumPy，输入data/粮食产量年度数据.csv；在本包目录从头顺序运行，题4—5依赖题3。输出到屏幕。本次练习评分不代表正式学期权重。


In [1]:
from pathlib import Path
import numpy as np
# Notebook请在本备课包文件夹启动；脚本以自身所在目录为基准。
BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA = BASE / "data" / "粮食产量年度数据.csv"
print("NumPy版本：", np.__version__)
print("数据文件存在：", DATA.exists())


NumPy版本： 2.3.5
数据文件存在： True


## 题1 数组入门（10分）
把模拟产量[100, 120, 110]转为float数组，输出shape、dtype、每项增加10后的结果；用一句话说明列表乘2与数组乘2的区别。


In [2]:
a = np.array([100, 120, 110], dtype=float)
print(a.shape, a.dtype, a + 10)


(3,) float64 [110. 130. 120.]


参考解释：形状(3,)，增加后[110,130,120]；列表乘2重复元素，数组乘2逐项翻倍。


## 题2 二维索引与axis（15分）
模拟表[[10,20],[30,40],[50,60]]，行代表三年，列代表甲乙两地，单位万吨。取乙地区全部产量，计算两地各自三年累计、每年的两地合计，并解释两个axis。


In [3]:
r = np.array([[10,20],[30,40],[50,60]])
print(r[:,1])
print(r.sum(axis=0))
print(r.sum(axis=1))


[20 40 60]
[ 90 120]
[ 30  70 110]


参考解释：乙地[20,40,60]；各地累计[90,120]，每年合计[30,70,110]。


## 题3 读取与筛选（20分）
读取配发CSV前两列，去掉年份的“年”，转换类型；同步按年份升序排列，再筛选2014—2023年。打印形状和首末年。提示：loadtxt、replace、astype、argsort、&。


In [4]:
# 文件有中文年份和表头，先按字符串只读前两列。
raw = np.loadtxt(DATA, delimiter=",", skiprows=1, usecols=(0, 1),
                 dtype=str, encoding="utf-8-sig")
year_list = []
for text in raw[:, 0]:
    year_list.append(int(text.replace("年", "")))
years_all = np.array(year_list)
yields_all = raw[:, 1].astype(float)
# 对年份排序后，产量必须用同一组位置重排，保持一一对应。
order = np.argsort(years_all)
years_all = years_all[order]
yields_all = yields_all[order]
print("形状：", raw.shape)
print("时间范围：", years_all[0], years_all[-1])
print("产量均为有限数：", np.all(np.isfinite(yields_all)))
# 明确筛选年份，不依赖文件前10行恰好是最近10年。
mask = (years_all >= 2014) & (years_all <= 2023)
years = years_all[mask]
yields = yields_all[mask]
print("年份：", years)
print("产量（万吨）：", yields)


形状： (75, 2)
时间范围： 1949 2023
产量均为有限数： True
年份： [2014 2015 2016 2017 2018 2019 2020 2021 2022 2023]
产量（万吨）： [63964.83 66060.27 66043.51 66160.73 65789.22 66384.34 66949.15 68284.75
 68652.77 69541.  ]


参考解释：全表形状(75,2)，1949—2023年；选定2014—2023年10条。


## 题4 统计与年份定位（20分）
针对题3的10年数据，求累计、平均、总体标准差(ddof=0)、最高最低产量与年份；输出高于均值的年份和数量。写清单位及统计范围。


In [5]:
mean_yield = np.mean(yields)
std_yield = np.std(yields, ddof=0)  # 描述选定10年的总体离散程度，分母为10。
print(f"2014—2023年累计产量：{np.sum(yields):.2f} 万吨")
print(f"平均年产量：{mean_yield:.2f} 万吨")
print(f"总体标准差：{std_yield:.2f} 万吨")
print("最高年份：", years[yields == np.max(yields)], "产量：", np.max(yields))
print("最低年份：", years[yields == np.min(yields)], "产量：", np.min(yields))
print("首个最大值的位置：", np.argmax(yields))
print(years[yields > mean_yield])
print(np.sum(yields > mean_yield))


2014—2023年累计产量：667830.57 万吨
平均年产量：66783.06 万吨
总体标准差：1546.95 万吨
最高年份： [2023] 产量： 69541.0
最低年份： [2014] 产量： 63964.83
首个最大值的位置： 9
[2020 2021 2022 2023]
4


参考解释：结果参见06_教师结果核对表.md；最高最低应以产量定位年份。


## 题5 增长与排名（20分）
计算2015—2023年增长量和同比增长率，输出下降年份；输出产量前5年。解释增长率分母、为什么10年只有9项变化、为什么不能对产量排序后计算同比。


In [6]:
# 时间已经升序；今年在后，上年在前，分母必须是上年产量。
growth = yields[1:] - yields[:-1]
growth_rate = growth / yields[:-1] * 100
growth_years = years[1:]
print("对应年份：", growth_years)
print("年增长量（万吨）：", np.round(growth, 2))
print("同比增长率（%）：", np.round(growth_rate, 2))
print("下降年份：", growth_years[growth < 0])
rank = np.argsort(yields)[::-1][:5]
print(years[rank], yields[rank])


对应年份： [2015 2016 2017 2018 2019 2020 2021 2022 2023]
年增长量（万吨）： [2095.44  -16.76  117.22 -371.51  595.12  564.81 1335.6   368.02  888.23]
同比增长率（%）： [ 3.28 -0.03  0.18 -0.56  0.9   0.85  1.99  0.54  1.29]
下降年份： [2016 2018]
[2023 2022 2021 2020 2019] [69541.   68652.77 68284.75 66949.15 66384.34]


参考解释：同比以上年为分母；首年缺窗口内的上年；排名顺序不是时间顺序。


## 题6 分析报告与提交（15分）
写150—250字：统计范围与单位、平均与最高年份、下降年份、2023年同比、至少一项数据局限。不得写成“逐年持续上升”，不得无依据归因。保留运行输出。可使用教师导出模板输出增长表。


In [7]:
print("报告示例见本单元下方；请结合输出检查。")


报告示例见本单元下方；请结合输出检查。


参考解释：2014—2023年的平均年产量约66783.06万吨，2023年产量最高，为69541万吨。2016年、2018年较上年下降，说明期末水平高于期初不等于逐年增长。2023年同比增长约1.29%。本报告仅使用教材配套文件的10个年度值，数据止于2023年，未重新核验发布来源和修订口径；不能据此判断当前产量、解释变化的因果或预测未来收益。


## 拓展题 不计基础分
复现模拟播种面积，说明随机种子的作用；计算1949—1958与2014—2023两期平均水平增幅，以及2014—2023年复合年均增长率，解释两种增长率为何不同。


In [8]:
rng = np.random.default_rng(42)
area_sim = np.round(rng.uniform(110000, 120000, size=10), 2)
print("模拟面积（千公顷）：", area_sim)
left = np.array([[1, 2], [3, 4]])
right = np.array([[2, 0], [0, 2]])
print("逐元素乘法：", left * right)
print("矩阵乘法：", left @ right)
early = yields_all[(years_all >= 1949) & (years_all <= 1958)]
late = yields_all[(years_all >= 2014) & (years_all <= 2023)]
print("最早10年平均年产量（万吨）：", early.mean())
print("最新10年平均年产量（万吨）：", late.mean())
print("两期平均水平增幅（%）：", (late.mean() / early.mean() - 1) * 100)
print("全75年累计产量（万吨）：", yields_all.sum())
print("全75年平均年产量（万吨）：", yields_all.mean())
# 2014到2023是9个年度间隔，不能用10作指数分母。
cagr = ((yields[-1] / yields[0]) ** (1 / (years[-1] - years[0])) - 1) * 100
print("2014—2023年复合年均增长率（%）：", cagr)


模拟面积（千公顷）： [117739.56 114388.78 118585.98 116973.68 110941.77 119756.22 117611.4
 117860.64 111281.14 114503.86]
逐元素乘法： [[2 0]
 [0 8]]
矩阵乘法： [[2 4]
 [6 8]]
最早10年平均年产量（万吨）： 16587.12
最新10年平均年产量（万吨）： 66783.057
两期平均水平增幅（%）： 302.61996657647626
全75年累计产量（万吨）： 2880143.98
全75年平均年产量（万吨）： 38401.919733333336
2014—2023年复合年均增长率（%）： 0.9330269838975447


参考解释：模拟面积不是实际面积；两期均值比较不考虑逐年路径，CAGR用首末值及9个年度间隔。
